In [5]:
import pandas as pd

rem = pd.read_excel("output_textmodel_ECB_v2.xlsx", sheet_name="remarks_textmodel")
qa  = pd.read_excel("output_textmodel_ECB_v2.xlsx", sheet_name="answers_textmodel")
rem["date"] = rem["date"].astype(str)
qa["date"]  = qa["date"].astype(str)

def gpt_tone(df):
    # TextSentiment = (dovish - hawkish) / (dovish + hawkish); neutrals excluded
    g = df.groupby("date")["label"]
    H = g.apply(lambda s: (s == "hawkish").sum())
    D = g.apply(lambda s: (s == "dovish").sum())
    return (D - H) / (H + D)

w_tone   = gpt_tone(rem).rename("w_tone")
qa_tone  = gpt_tone(qa).rename("qa_tone")
pooled   = pd.concat([rem[["date", "label"]], qa[["date", "label"]]])
all_tone = gpt_tone(pooled).rename("all_tone")

tone = pd.concat([w_tone, qa_tone, all_tone], axis=1).reset_index().sort_values("date")

out = "output_textmodel_ECB_v2_conference.xlsx"
with pd.ExcelWriter(out, engine="openpyxl") as writer:
    tone.to_excel(writer, index=False)
